## [节点配置](https://docs.langchain.com/oss/javascript/langgraph/graph-api-nodes)

### [异常处理](https://docs.langchain.com/oss/python/langgraph/fault-tolerance#error-handling)

In [3]:
import operator
import time
from typing import Annotated

from langgraph.cache.memory import InMemoryCache
from langgraph.graph import StateGraph, START, END
from langgraph.types import CachePolicy
from loguru import logger
from typing_extensions import TypedDict


class State(TypedDict):
    user: str
    invoke_counts: Annotated[int, operator.add]

def expensive_node(state: State):
    logger.info(f"节点被调用 user: {state['user']}")
    time.sleep(3)
    logger.info(f"节点执行完成")
    return {
        "invoke_counts": 1,
    }


builder = StateGraph(State)
builder.add_node(
    expensive_node,
    cache_policy=CachePolicy(ttl=10),
)
builder.add_edge(START, "expensive_node")
builder.add_edge("expensive_node", END)
graph = builder.compile(cache=InMemoryCache())

logger.info(graph.invoke({"user": "max", "invoke_counts": 0}, stream_mode="updates"))
logger.info(graph.invoke({"user": "max", "invoke_counts": 0}, stream_mode="updates"))

2026-11-07 20:45:17.841 | INFO     | __main__:expensive_node:17 - 节点被调用 user: max
2026-11-07 20:45:20.847 | INFO     | __main__:expensive_node:19 - 节点执行完成
2026-11-07 20:45:20.850 | INFO     | __main__:<module>:34 - [{'expensive_node': {'invoke_counts': 1}}]
2026-11-07 20:45:20.852 | INFO     | __main__:<module>:35 - [{'expensive_node': {'invoke_counts': 1}, '__metadata__': {'cached': True}}]
